# 🔍 Raw CSV Inspection & Pre-Merge Checks

Inspects each of the 4 raw CSV files in `data/raw/` (`owid_co2.csv`, `owid_energy.csv`, `wb_gdp_pc.csv`, `wb_gdp_total.csv`):
- Prints all column names and dtypes
- Counts `NaN` / null values per column
- Checks join keys, duplicate rows, year ranges, and overlapping column names before merging into a single dataset

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 50)

RAW_DIR = Path('../data/raw')
FILES = ['owid_co2.csv', 'owid_energy.csv', 'wb_gdp_pc.csv', 'wb_gdp_total.csv']

dfs = {}
for fname in FILES:
    path = RAW_DIR / fname
    df = pd.read_csv(path)
    # Handle World Bank bulk CSVs that start with 4 metadata rows
    if len(df.columns) <= 2:
        df = pd.read_csv(path, skiprows=4)
    dfs[fname] = df
    print(f'Loaded {fname:<20} -> shape: {df.shape}')

## 1. Per-CSV Column List, Data Types & NaN / Null Summary

In [ ]:
for fname, df in dfs.items():
    print('=' * 80)
    print(f'FILE: {fname}  |  Rows: {len(df):,}  |  Columns: {df.shape[1]}')
    print('Column names:', list(df.columns))
    
    null_summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notna().sum(),
        'null_count': df.isna().sum(),
        'null_%': (df.isna().mean() * 100).round(2),
    })
    display(null_summary)
    display(df.head(3))

## 2. Pre-Merge Readiness Checks Across All 4 CSVs

In [ ]:
# Check 1: Overlapping column names across files (potential _x / _y merge collisions)
print('--- Overlapping columns across CSVs ---')
all_cols = {fname: set(df.columns) for fname, df in dfs.items()}
for i, f1 in enumerate(FILES):
    for f2 in FILES[i+1:]:
        shared = sorted(all_cols[f1] & all_cols[f2])
        print(f'{f1}  ∩  {f2}: {shared}')

# Check 2: Join keys, nulls in keys, duplicate rows, and year coverage
print('\n--- Key integrity & year coverage per CSV ---')
for fname, df in dfs.items():
    print(f'\n[{fname}]')
    print(f'  Exact duplicate rows: {df.duplicated().sum()}')
    for key_col in ['country', 'iso_code', 'Country Name', 'Country Code', 'economy', 'year', 'time']:
        if key_col in df.columns:
            print(f'  Key "{key_col}": {df[key_col].isna().sum()} nulls | {df[key_col].nunique()} unique')
    if 'year' in df.columns:
        print(f'  Year range: {df["year"].min()} to {df["year"].max()}')
    if 'iso_code' in df.columns and 'year' in df.columns:
        dups = df.dropna(subset=['iso_code']).duplicated(subset=['iso_code', 'year']).sum()
        owid_aggs = df['iso_code'].dropna().str.startswith('OWID').sum()
        print(f'  Duplicate (iso_code, year) pairs: {dups} | OWID_* aggregate rows: {owid_aggs}')

## 3. Check Country / ISO Code Overlap Before Merging

In [ ]:
iso_sets = {}
for fname, df in dfs.items():
    iso_col = next((c for c in ['iso_code', 'Country Code', 'economy'] if c in df.columns), None)
    if iso_col:
        iso_sets[fname] = set(df[iso_col].dropna().unique())
        print(f'{fname:<20} ({iso_col}): {len(iso_sets[fname])} unique codes')

if len(iso_sets) >= 2:
    common_isos = set.intersection(*iso_sets.values())
    print(f'\nCommon ISO codes present in ALL files: {len(common_isos)}')
    for fname, codes in iso_sets.items():
        only_here = sorted(codes - common_isos)
        print(f'  In {fname} but not in all ({len(only_here)}): {only_here[:15]}')